In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

doc_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Article-Encoder")
doc_model = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder")

query_tokenizer = AutoTokenizer.from_pretrained(
    "ncbi/MedCPT-Query-Encoder")
query_model = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder")


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        1
    ) / torch.clamp(
        input_mask_expanded.sum(1),
        min=1e-9
    )


def encode_document(text):

    encoded_input = doc_tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    )

    with torch.no_grad():

        model_output = doc_model(
            **encoded_input
        )

    embeddings = mean_pooling(
        model_output,
        encoded_input["attention_mask"]
    )

    return embeddings[0]


def encode_query(query):

    encoded_input = query_tokenizer(
        query,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=512
    )

    with torch.no_grad():

        model_output = query_model(
            **encoded_input
        )

    embeddings = mean_pooling(
        model_output,
        encoded_input["attention_mask"]
    )

    return embeddings[0]

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5704.86it/s]


In [3]:
query = "Does amyloid beta contribute to cognitive decline?"
doc = """
<b>INTRODUCTION</b>: Despite expanding use of plasma biomarkers for Alzheimer's disease (AD), minimal examination has been undertaken in early-onset AD (EOAD). Prior analyses assessed criterion and convergent validity of common plasma biomarkers in a well-characterized sample with sporadic EOAD. <b>METHODS</b>: Plasma amyloid beta (A\u03b2) 42/40, tau phosphorylated at threonine 231 (p-tau231), glial fibrillary acidic protein (GFAP), and neurofilament light change (NfL) levels were obtained for 189 EOAD, 52 early-onset non-AD (EOnonAD), and 83 cognitively normal (CN) participants. Diagnostic group differences, convergence with imaging biomarkers, and the capacity to predict specific cognitive domains were investigated. <b>RESULTS</b>: After controlling for cognitive status, EOAD participants exhibited more pathologic levels of A\u03b242/40, p-tau231, and GFAP relative to EOnonAD participants. However, NfL displayed the greatest sensitivity to non-amnestic cognitive impairment across the sample. <b>DISCUSSION</b>: These results establish criterion and convergent validity of plasma biomarkers in sporadic EOAD and highlight complementary roles that AD-specific and non-specific plasma markers may play across the course of AD care.
"""

q_emb = encode_query(query)

d_emb = encode_document(doc)

similarity = torch.nn.functional.cosine_similarity(
    q_emb,
    d_emb,
    dim=0
)

print(similarity.item())


0.781026303768158


In [4]:
print(q_emb.shape)

torch.Size([768])
